In [0]:
df = spark.sql("""
create or replace temp view vw_gold_claims_by_policy_type_and_status
AS
SELECT
    policy_type,
    claim_status,
    COUNT(*) AS total_claims,
    SUM(claim_amount) AS total_claim_amount
FROM
    policyprojcatalog.silver.claim c
JOIN policyprojcatalog.silver.policy p
    ON c.policy_id = p.policy_id
GROUP BY
    policy_type,
    claim_status
HAVING p.policy_type IS NOT NULL
""")
df.display()


In [0]:
%sql
MERGE INTO policyprojcatalog.gold.sales_by_policytype_status AS T
USING vw_gold_claims_by_policy_type_and_status AS S
ON T.policy_type = S.policy_type
AND T.claim_status = S.claim_status

WHEN MATCHED THEN
UPDATE SET
    T.total_claim_amount = S.total_claim_amount,
    T.total_claim = S.total_claims,
    T.updated_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    policy_type,
    claim_status,
    total_claim_amount,
    total_claim,
    updated_timestamp
)
VALUES (
    S.policy_type,
    S.claim_status,
    S.total_claim_amount,
    S.total_claims,
    current_timestamp()
);

In [0]:
%sql

select * from policyprojcatalog.gold.sales_by_policytype_status

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_gold_claims_analysis AS
SELECT
    policy_type,
    AVG(claim_amount) AS avg_claim_amount,
    MAX(claim_amount) AS max_claim_amount,
    MIN(claim_amount) AS min_claim_amount,
    COUNT(DISTINCT claim_id) AS total_claims
FROM
    policyprojcatalog.silver.claim c
JOIN policyprojcatalog.silver.policy p
    ON c.policy_id = p.policy_id
GROUP BY
    policy_type
HAVING p.policy_type IS NOT NULL;

In [0]:
%sql
MERGE INTO policyprojcatalog.gold.claim_analysis AS T
USING vw_gold_claims_analysis AS S
ON T.policy_type = S.policy_type

WHEN MATCHED THEN
UPDATE SET
    T.avg_claim_amount = S.avg_claim_amount,
    T.max_claim_amount = S.max_claim_amount,
    T.min_claim_amount = S.min_claim_amount,
    T.total_claims = S.total_claims,
    T.updated_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    policy_type,
    avg_claim_amount,
    max_claim_amount,
    min_claim_amount,
    total_claims,
    updated_timestamp
)
VALUES (
    S.policy_type,
    S.avg_claim_amount,
    S.max_claim_amount,
    S.min_claim_amount,
    S.total_claims,
    current_timestamp()
);

In [0]:
%sql
select * from policyprojcatalog.gold.claim_analysis

In [0]:
%sql
select * from policyprojcatalog.gold.sales_by_policytype_status

In [0]:
%sql
select * from policyprojcatalog.gold.sales_by_policytype_month